In [ ]:
import numpy as np
from scipy.special import iv
from scipy.integrate import quad
import matplotlib.pyplot as plt
from warnings import filterwarnings
filterwarnings("ignore")  

rng = np.random.default_rng(seed=0)

*Our goal with Monte Carlo is to sample from a posterior distribution, so we can estimate whichever statistic we want to. As we have seen in the previous two chapters, sampling from the posterior can be quite difficult. It would be nice to come up with something a bit more general than what we have seen.* 

*Before we jump into MCMC, I'm always of the opinion that whenever you see a new technique, it's best to retrace the steps of the inventors of said technique, almost to feel like, if you were in their shoes, you could have invented this yourself.* 

# Statistical Mechanics and the Metropolis-Hastings Algorithm
MCMC was invented, alongside the development of the early computer, to solve problems relating to nuclear physics, by Nicolas Metropolis. Him, and some familiar names like John Von Neumann and Stanislaw Ulam, had invented Monte Carlo techniques a few years back in 1946 to solve probability questions about Solitaire (Ulam had initially proposed Monte Carlo), and they been working on inventing some important techniques like inverse transform sampling, and accept-reject sampling. However, they had been running into the same problems we've seen in the previous chapters. Let's introduce the problems they were working on. 

Suppose you have $N$ particles, $\theta_1, \ldots, \theta_N$, each living in $\mathbb{R}^2$, and you have some real-valued potential energy $V$, which is a function of distance between particles. We would then have our energy in the following form:
$$
E = \frac12 \sum_{i \neq j} V (\|\theta_i - \theta_j\|)
$$
The real world chooses configurations of $\theta = (\theta_1, \ldots, \theta_N)$ that minimizes energy. To be more precise, configurations, or states, of lower energy should be of higher likelihood. I.e. $-E(\theta)$ is our *score* function we seek to maximize. We want to understand the likely states, so we need a probability distribution over the possible states. As some astute readers may be thinking, an appropriate thing to do is to take a *softmax* over the possible configurations! That's exactly what physicists do. We define the *Boltzmann distribution* to have density 
$$
\pi(\theta) \propto \exp(\beta s(\theta)) = \exp(- E(\theta) / \kappa T)
$$
Where $s(\theta) = -E(\theta)$ is a score function, $\kappa$ is the Boltzmann constant, $T$ is temperature, and $\beta$ is an inverse-temperature parameter. As such, these scientists were interested in computing integrals of the form 
$$
\int F(\theta) \pi(\theta) d\theta
$$
where $F$ is some statistic of interest. We can already see the parallels of the problems we faced in the previous part. Clearly, analytical techniques are intractable, since $N$ is too large. The likelihood function is extremely peaked, making naive monte carlo very difficult to do. Importance sampling was invented by this time, for use in this problem as well. However, Metropolis' idea was very powerful as well. 

Consider an arbitrary intial distribution of $\theta$. Take some $\theta_i$, and perturb it randomly. Precisely, we take $\xi_1$ and $\xi_2$ to be i.i.d $\mathrm{Unif}[-1,1]$ random variables, and map $\theta_i \mapsto \theta_i + \alpha(\xi_1, \xi_2)$. However, we could have taken any reasonable distribution for the perturbation. Now, we recompute the energy of the system, and find what the change in energy $\Delta E$ is. If $\Delta E < 0$, then the move puts the system into a state of lower energy, and so we allow the move. However, if $\Delta E > 0$, we may also allow the move. We compare the likelihoods of the two states $\theta$ and $\theta'$, and accept the move with some corresponding probability. For instance, if a particular move increases the energy, but the likelihood of the new state is not too much lower than the likelihood of the old state, then it makes sense that the state could randomly change to the higher energy state. How likely should it be to move to the new state? That is a good question, it's not entirely clear yet. Let's just set the probability to be some function of the likelihoods of each of the state, i.e. $\rho(\theta, \theta')$, and we will solve for it later. How are we going to do that? Well first, you might notice that this process defines a Markov Chain!

One important fact about Markov Chains is that if you run them long enough, if they are "nice" enough, they will converge to some stationary distribution. The idea of the Markov chain we set up above is that after a great number of iterations, the particles will settle upon some physical equilibrium, and this equilibrium is exactly the Gibbs Distribution we are trying to analyze. Thus, the stationary distribution will be the Gibbs distribution we are looking to sample from. So now we have an interesting candidate for sampling from this Gibbs distribution. 

1) Define an arbitrary state
2) Run the process above until it converges
3) Obtain the sample, and start from the top

Once we do this enough times, we get our Monte Carlo estimate. This is already pretty good, but some basic Markov Chain theory makes this approach even better. Instead of having to restart the chain for every fresh sample, something called the Ergodic Theorem tells us is that if we have a "nice enough" markov chain $X_t$, and its corresponding statinoary distribution $\mu$, then for any two functions $f$ and $g$, we have:
$$
\frac{\sum_{t = 1}^T f(X_t)}{\sum_{t = 1}^T g(X_t)} \to \frac{\int f d \mu}{\int g d \mu}
$$
This basically tells us that we don't need to restart the Markov chain for each sample, we just keep applying the procedure above and each new $\theta'$ that we get will suffice for our Monte Carlo estimate. This is pretty fantastic, however, we still need to figure out what $\rho$ we should take to guarantee that our Markov chain has the right stationary. To find it, suppose we have $\theta \sim \pi(\theta)$.  Then, letting $Q$ summarize the procedure for generating the next step, if $Q(\theta)$ is also distributed $\pi(\theta)$, then we have found the correct $\rho$. So, let's compute the distribution of $Q(\theta)$. Let $A$ be a borel set, and let's abstract the uniform random walk to some arbitrary distribution $q(\theta' | \theta)$, i.e. we go from $\theta$ to $\theta'$ with density $q(\theta'|\theta)$. Conditioning on the draw of $\theta$ from $\pi(\theta)$, we have 
\begin{align*}
\Pr(\theta' \in A | \theta) &= \int \Pr(\theta' \in A | \theta, \theta') q(\theta' | \theta) d \theta' = \int (\Pr(\theta' \in A | \theta, \theta', \Delta E < 0)[\Delta E < 0] + \Pr(\theta' \in A | \theta, \theta', \Delta E> 0)[\Delta E < 0]) q(\theta'|\theta) d \theta\\
&= \int [\theta' \in A, \Delta E < 0] q(\theta'| \theta) + ([\theta' \in A, \Delta E > 0] \rho(\theta, \theta') + [\theta \in A, \Delta E > 0] (1 - \rho(\theta, \theta'))) q(\theta'|\theta) d \theta'
\end{align*}
Now, we have to find the unconditional distribution of $\theta'$, so we integrate over $\theta$. 
\begin{align*}
\Pr(\theta'\in A) &= \iint \bigg([\theta' \in A, \Delta E < 0] + [\theta' \in A, \Delta E > 0] \rho(\theta, \theta') + [\theta \in A, \Delta E > 0] (1 - \rho(\theta, \theta'))\bigg) q(\theta'|\theta) \pi(\theta) d\theta'd\theta\\
\end{align*}
We get 3 separate integrals. Let's focus on each at a time:

\begin{align}
\iint [\theta' \in A, \Delta E < 0] q(\theta'|\theta) \pi(\theta) d\theta'd\theta \\
\iint [\theta' \in A, \Delta E > 0] \rho (\theta, \theta') q(\theta'|\theta) \pi(\theta) d \theta'd\theta\\
\iint [\theta \in A, \Delta E > 0] (1 - \rho(\theta, \theta')) q(\theta' | \theta)\pi(\theta) d\theta' d\theta
\end{align}

Notice that we can swap $\theta$ and $\theta'$ to get, for $(3)$
$$
\iint [\theta \in A, \Delta E > 0]q(\theta'|\theta)\pi(\theta)d\theta'd\theta - \iint [\theta' \in A, \Delta E < 0] \rho(\theta', \theta) q(\theta | \theta') \pi(\theta') d\theta'd \theta
$$
and for $(2)$
$$
\iint [\theta \in A, \Delta E < 0] \rho(\theta', \theta) q(\theta | \theta') \pi(\theta') d\theta' d \theta
$$
Grouping, we get
$$
\iint [\theta' \in A, \Delta E < 0] (q(\theta'|\theta) \pi(\theta) - \rho(\theta', \theta)q(\theta | \theta') \pi (\theta')) d\theta' d\theta
$$
and 
$$
\iint [\theta \in A, \Delta E > 0] q(\theta' | \theta) \pi(\theta) + [\theta \in A, \Delta E < 0] \rho(\theta', \theta)q(\theta|\theta')\pi(\theta')d\theta'd\theta
$$
Now, we notice something interesting! If
$$
q(\theta'|\theta)\pi(\theta) = \rho(\theta', \theta) q(\theta|\theta')\pi(\theta')
$$
then not only does the first integral disappear, but the second integral nicely simplifies to 
$$
\iint \bigg([\theta \in A, \Delta E > 0] + [\theta \in A, \Delta E < 0] \bigg) q(\theta' | \theta) d\theta'd\theta = \int_A \int q(\theta'|\theta) d\theta' \pi(\theta) d\theta = \int_A \pi(\theta) d\theta
$$
So, once we choose 
$$
\rho(\theta', \theta) = \frac{q(\theta' | \theta) \pi(\theta)}{q(\theta|\theta')\pi(\theta')}
$$
We get that the stationary distribution of our Markov Chain is precisely $\pi(\theta)$. Additionally, we notice that this simplifies a bit, since we have a ratio of likelihoods. Writing this in the form we defined $\rho(\theta, \theta')$, we have:
$$
\rho(\theta, \theta') = \exp(- \Delta E / \kappa T) \frac{q(\theta|\theta')}{q(\theta'|\theta)}
$$
which avoids computing the normalization factor of $\pi(\theta)$. In the case of the uniform perturbation, the second term cancels out. 
The astute reader will note that we can get rid of the extra step of checking if $\Delta E < 0$, by simply taking the minimum of $\rho$ with 1. Then, we have a slightly simplified procedure. 

This derivation was a little abstraction heavy, however, it has set us up nicely. For instance, now suppose instead of working on a statistical mechanics problem, we have a Bayesian problem. We have a prior $\pi(\theta)$, and a likelihood function $L(x | \theta)$. Very often, we are in the same boat Metropolis was in. We know $\pi(\theta | x)$ up to a normalization constant. Specifically we know
$$
\pi(\theta | x) \propto L(x | \theta) \pi(\theta)
$$
We can apply the same solution! If we use some $q(\theta' | \theta)$ whose support covers $\pi(\theta' | x)$'s support, for every $\theta$, then we choose 
$$
\rho = \frac{q(\theta | \theta') L(x | \theta') \pi(\theta')}{q(\theta' | \theta) L(x | \theta) \pi(\theta)} \land 1
$$
And we get back a Markov chain whose stationary distribution is $\pi(\theta | x)$. This is the Metropolis-Hasting algorithm, and it is the cornerstone of Markov Chain Monte Carlo. We summarize the algorithm below:

1) Sample $\theta^{(0)}$ from an arbitrary distribution.
2) Sample $\theta$ from $q(\theta | \theta^{(n)})$. 
3) Compute $$\rho = \frac{q(\theta^{(n)} | \theta) L(x | \theta) \pi(\theta)}{q(\theta | \theta^{(n)}) L(x | \theta^{(n)}) \pi(\theta^{(n)})} \land 1$$
4) Set $$\theta^{(n+1)} = \begin{cases} \theta & \text{w.p. } \rho, \\ \theta^{(n)} & \text{w.p. } 1 - \rho\\ \end{cases} $$

There are a few technical conditions to guarantee that this algorithm actually produces a markov chain that converges, but the only really important one is that the support of $\pi(\theta | x)$ is contained in the support of $\theta'\mapsto q(\theta' | \theta)$. Theoretically, almost any $q$ will work; however, in practice, a bad overlap between $q$ and $\pi(\theta | x)$ will seriously hurt performance. However, there has been a huge host of different ways to choose $q$. In the coming sections, we will investigate the different ways to choose $q$. 